# Welcome to the INTSYCURE Jupyter Notebook

This will be the primary jupyter notebook for us to do our feature engineering and model training. Kindly follow each section when modifying this notebook to make things easier. Feel free to make subsections if needed.

## README

This code's main branch will be in `mco2`. For creating your own branch, kindly format the branch name with `mco2-<name>` (first name or last name up to you). Branch off from this `mco2` branch, make your changes, and the changes can be later pulled into the main branch or specific features/code can be manaully taken from each of our notebooks, since the jupyter notebook format may be a pain in the azz to work with in git.

## Import Modules

Kindly install these modules if required.

In [1]:
# Installing packages (run in cli):
#   pip install <package name 

import pandas as pd
import pickle

## Importing the Data
The `master_dataset.csv` file is imported, ready to be used.

In [2]:
master_data = pd.read_csv('master_dataset.csv')
master_data.head

<bound method NDFrame.head of       sentence_id  word_id                                           sentence  \
0               0        0  Kaya kayong mga babae wag kayong basta basta m...   
1               0        1  Kaya kayong mga babae wag kayong basta basta m...   
2               0        2  Kaya kayong mga babae wag kayong basta basta m...   
3               0        3  Kaya kayong mga babae wag kayong basta basta m...   
4               0        4  Kaya kayong mga babae wag kayong basta basta m...   
...           ...      ...                                                ...   
9631          499     9631  Hello po mag ask po ako sa inyo ng help para p...   
9632          499     9632  Hello po mag ask po ako sa inyo ng help para p...   
9633          499     9633  Hello po mag ask po ako sa inyo ng help para p...   
9634          499     9634  Hello po mag ask po ako sa inyo ng help para p...   
9635          499     9635  Hello po mag ask po ako sa inyo ng help para p...  

## Feature Engineering

In this section, we will engineer each of the features for the model to be trained.

### Feature Tracking
Consider this as a checklist of the features that I am considering
- [x] has consecutive a/i/u  ; ENG words often only have consecutive Os or Es but not usually A, I, or U
- [x] has x  ; does not appear often in FIL words
- [x] has z  ; does not appear often in FIL words
- [x] has a repeating prefix  ; i.e. **kaka**in
- [x] has a repeating first char  ; i.e. **uupo**
- [x] has number  ; bc all numbers SHOULD be OTH (NOTE: we will probably have to clean the data for this)
- [x] has special char  ; bc all numbers SHOULD be OTH (NOTE: we will probably have to clean the data for this)

In [3]:
# creating a copy of the master dataset for feature
df = master_data.copy()

import extractor

%load_ext autoreload
%autoreload 2

### Has Consecutive A/I/U

English words may have consecutive Os (boom) or Es (tree) but not usually A, I, or U. Filipino may have words such as **kaakbay** (consecutive As) **ginigiit** (consecutive Is) or **uupo** (consecutive Us)

In [4]:
df = extractor.extract_has_consec_aui(df)

### Has X, Has Z
Both X and Z are not common or are not present at all in Filipino words.

In [5]:
df = extractor.extract_has_xz(df)

### Has Repeating prefix
Looks for words with a repeating start i.e. **kakakain** or **mamaya**

NOTE: does not cover words with a repeating first letter. That will be a separate feature to clarify their difference. This will also only check for prefixes with length 2-4.

In [6]:
df = extractor.extract_repeating_prefix(df)

### Has Repeating First Letter
Looks for words with a repeating first letter i.e. **uupo**

In [7]:
df = extractor.extract_repeating_first_letter(df)

### Has Number or Special Character
Looks for any number in the word. Another feature is for any special character being in the word.

In [8]:
df = extractor.extract_has_num_or_spec_char(df)

## Model Training
This section will contain the main code for training and exporting our model.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB, MultinomialNB, ComplementNB

from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

In [10]:
# view the features 
print(df.columns)

Index(['sentence_id', 'word_id', 'sentence', 'word', 'annot', 'consec_aui',
       'has_x', 'has_z', 'rpt_prfx', 'rpt_fchr', 'has_num', 'has_spec'],
      dtype='object')


### Splitting Training Data

In [11]:
# excluding word col for now, to allow training with the tfid vectorizer
feature_cols = [c for c in df.columns if c not in ['sentence_id', 'word_id', 'sentence', 'annot']]

X = df[feature_cols]
y = df['annot'].to_numpy()

rand_state_a = 548193
rand_state_b = 395813

x_train, x_test, y_train, y_test = train_test_split(X, y, train_size=0.7, test_size=0.3, random_state=rand_state_a)
x_test, x_val, y_test, y_val = train_test_split(x_test, y_test, test_size=0.5, random_state=rand_state_b)

### Training the Model

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
char_vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2,5), max_features=20000)


In [13]:
from scipy.sparse import hstack, csr_matrix

model = ComplementNB()

x_tfidf_train = char_vectorizer.fit_transform(x_train['word'])

display(x_train)
x_ftrs_train = csr_matrix(x_train.drop('word', axis=1).values)

x_combined_train = hstack([x_ftrs_train, x_tfidf_train])

model.fit(x_combined_train, y_train)

,word,consec_aui,has_x,has_z,rpt_prfx,rpt_fchr,has_num,has_spec
6030,eto,False,False,False,False,False,False,False
9112,ng,False,False,False,False,False,False,False
9454,May,False,False,False,False,False,False,False
2870,sa,False,False,False,False,False,False,False
6902,mga,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
6616,lets,False,False,False,False,False,False,False
1588,mo,False,False,False,False,False,False,False
9463,ng,False,False,False,False,False,False,False
4744,naming,False,False,False,False,False,False,False


,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueOnly used in edge case with a single class in the training set.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. Not used.",None
,"norm norm: bool, default=FalseWhether or not a second normalization of the weights is performed. Thedefault behavior mirrors the implementations found in Mahout and Weka,which do not follow the full algorithm described in Table 9 of thepaper.",False
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](4,)","[ 23., 624.,5141., 957.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class. Only used in edgecase with a single class in the training set.","ndarray[float64](4,)","[-5.68,-2.38,-0.27,-1.95]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U3](4,)","['CS','ENG','FIL','OTH']"
"feature_all_ feature_all_: ndarray of shape (n_features,)Number of samples encountered for each feature during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](12741,)","[87. ,20. ,21. ,..., 0.2 , 0.34, 0.34]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature) during fitting.This value is weighted by the sample weight when provided.","ndarray[float64](4, 12741)","[[ 1. , 2. , 1. ,..., 0. , 0. , 0. ], [ 0. ,12. , 8. ,..., 0. , 0. , 0. ], [81. , 1. , 0. ,..., 0. , 0. , 0. ], [ 5. , 5. ,12. ,..., 0.2 , 0.34, 0.34]]"


### Model Evaluation

In [14]:
x_tfidf_test = char_vectorizer.transform(x_test['word'])
x_ftrs_test = csr_matrix(x_test.drop('word', axis=1).values)
x_combined_test = hstack([x_ftrs_test, x_tfidf_test])

y_pred = model.predict(x_combined_test)



display(f1_score(y_test, y_pred, average='weighted'))

display(confusion_matrix(y_test, y_pred, labels=['ENG', 'FIL','OTH', 'CS']))

0.923633580272767

array([[ 106,   18,    6,    5],
       [  23, 1035,    6,    8],
       [  21,   22,  189,    1],
       [   1,    3,    1,    0]])

### Exporting the Model (and the vectorizer)

In [15]:
# commenting these out bc they are obsolete
# pickle.dump(model, open('pinoybot_model.pk1', 'wb'))
# pickle.dump(char_vectorizer, open('pinoybot_vectorizer.pk1', 'wb'))

## Test the Importing of the Model


In [16]:
# commenting these out bc they are obsolete
# model = pickle.load(open('pinoybot_model.pk1', 'rb'))
# vectorizer = pickle.load(open('pinoybot_vectorizer.pk1', 'rb'))

# x_tfidf_test = vectorizer.transform(x_test['word'])
# x_ftrs_test = csr_matrix(x_test.drop('word', axis=1).values)
# x_combined_test = hstack([x_ftrs_test, x_tfidf_test])

# y_pred = model.predict(x_combined_test)

# display(f1_score(y_test, y_pred, average='weighted'))
# print("If you saw an output and it's the same fscore as the one above.")

## Model Training Pipeline

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

model_pipeline = Pipeline([
    ('feature_extraction', FunctionTransformer(extractor.extract_features)), 
    ('tfidf', ColumnTransformer(
        transformers=[
            ('char_tfidf', TfidfVectorizer(analyzer='char', ngram_range=(2,5), max_features=20000), 'word'),
        ],
        remainder='passthrough'
    )),
    ('training', ComplementNB())
])

df = master_data.copy()

feature_cols = [c for c in df.columns if c not in ['sentence_id', 'word_id', 'sentence', 'annot']]

X = df[feature_cols]
y = df['annot'].to_numpy()

rand_state_a = 548193
rand_state_b = 395813

x_train, x_test, y_train, y_test = train_test_split(X, y, train_size=0.7, test_size=0.3, random_state=rand_state_a)
x_test, x_val, y_test, y_val = train_test_split(x_test, y_test, test_size=0.5, random_state=rand_state_b)

model_pipeline.fit(x_train, y_train)

y_pred = model_pipeline.predict(x_test)
display(f1_score(y_test, y_pred, average='weighted'))

pickle.dump(model_pipeline, open('pinoybot_model_pipeline.pk1', 'wb'))

0.9228813013046254

In [18]:
model_import = pickle.load(open('pinoybot_model_pipeline.pk1', 'rb'))

y_pred = model_import.predict(x_test)
display(f1_score(y_test, y_pred, average='weighted'))

0.9228813013046254